# Phase 2 — Notebook 06: Ingestion End-to-End

**Goal:** Run the full ingestion pipeline on a real document — parse → chunk → embed → store. Inspect all stored rows and verify the output is complete and consistent.

**Prerequisites:** Notebooks 01–05 completed successfully.

## 1. Setup

In [ ]:
import sys, uuid, time, hashlib, json
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from core.config import get_settings
from core.constants import (
    CHUNK_MIN_TOKENS, CHUNK_MAX_TOKENS,
    MODALITY_TEXT, MODALITY_IMAGE, MODALITY_TABLE,
    RELATION_CHUNK_TO_ASSET,
)
settings = get_settings()
print("Settings loaded")

## 2. Load Models & Connections

In [ ]:
import psycopg, numpy as np
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer
import tiktoken

# Embedding model
model = SentenceTransformer(settings.embedding_model)
enc   = tiktoken.get_encoding("cl100k_base")

# DB connection
conn_url = settings.database_url \
    .replace("postgresql+asyncpg://", "postgresql://") \
    .replace("postgresql+psycopg://", "postgresql://")
conn = psycopg.connect(conn_url)
register_vector(conn)

print("Model + DB ready")

## 3. Step 1 — Parse Document

Using a synthetic document to keep the notebook self-contained. Replace with real file parsing from Notebook 01 for production runs.

In [ ]:
# Synthetic parsed document — mimic output from Notebook 01
parsed_doc = {
    "title": "Introduction to Retrieval-Augmented Generation",
    "source": "notebook_06_e2e",
    "doc_type": "text",
    "sections": [
        {
            "title": "Overview",
            "level": 1,
            "blocks": [
                "Retrieval-augmented generation (RAG) is a technique that combines information retrieval with language model generation.",
                "RAG systems retrieve relevant documents from a knowledge base and use them as context for the language model.",
                "This approach improves factual accuracy and reduces hallucination compared to pure generation.",
            ],
        },
        {
            "title": "Vector Search",
            "level": 1,
            "blocks": [
                "Vector search converts text into dense numerical representations called embeddings.",
                "Similar texts produce embeddings that are close together in the vector space.",
                "pgvector extends PostgreSQL with VECTOR data type and approximate nearest neighbour search.",
                "The ivfflat index partitions the vector space into lists for efficient ANN retrieval.",
            ],
        },
        {
            "title": "Hybrid Retrieval",
            "level": 1,
            "blocks": [
                "Hybrid retrieval combines dense vector search with sparse keyword search (BM25).",
                "Dense retrieval excels at semantic matching while BM25 captures exact keyword matches.",
                "Score normalisation is required before fusing dense and sparse scores.",
            ],
        },
    ],
    "images": [
        {"caption": "Figure 1: RAG system architecture showing retrieval and generation components."},
        {"caption": "Figure 2: Hybrid retrieval pipeline combining dense and sparse search."},
    ],
}

print(f"Document    : {parsed_doc['title']}")
print(f"Sections    : {len(parsed_doc['sections'])}")
print(f"Total blocks: {sum(len(s['blocks']) for s in parsed_doc['sections'])}")
print(f"Images      : {len(parsed_doc['images'])}")

## 4. Step 2 — Chunk

In [ ]:
def count_tokens(text):
    return len(enc.encode(text))

def semantic_chunk(blocks, embeddings, threshold):
    """Split blocks into chunks at semantic boundaries."""
    chunks, current = [], [blocks[0]]
    for i in range(1, len(blocks)):
        sim = float(np.dot(embeddings[i - 1], embeddings[i]))
        if sim < threshold:
            chunks.append(" ".join(current))
            current = [blocks[i]]
        else:
            current.append(blocks[i])
    chunks.append(" ".join(current))
    return chunks

all_chunks = []  # list of {content, section_title, section_level, token_count}

for section in parsed_doc["sections"]:
    blocks = section["blocks"]
    if len(blocks) == 1:
        chunks = blocks
    else:
        block_embeddings = model.encode(blocks, normalize_embeddings=True)
        chunks = semantic_chunk(blocks, block_embeddings, settings.chunking_similarity_threshold)

    for chunk_text in chunks:
        tokens = count_tokens(chunk_text)
        all_chunks.append({
            "content": chunk_text,
            "section_title": section["title"],
            "section_level": section["level"],
            "token_count": tokens,
        })

print(f"Total chunks produced: {len(all_chunks)}")
for i, c in enumerate(all_chunks):
    print(f"  [{i+1}] [{c['section_title']}] {c['token_count']} tokens — {c['content'][:60]}...")

## 5. Step 3 — Embed Chunks

In [ ]:
t0 = time.time()
contents = [c["content"] for c in all_chunks]
chunk_embeddings = model.encode(
    contents,
    normalize_embeddings=True,
    batch_size=settings.embedding_batch_size,
)
print(f"Embedded {len(chunk_embeddings)} chunks in {time.time() - t0:.2f}s")
print(f"Shape: {chunk_embeddings.shape}")

## 6. Step 4 — Store to Postgres

In [ ]:
doc_id = str(uuid.uuid4())
section_id_map = {}   # section_title → section_id

with conn.cursor() as cur:
    # Insert document
    cur.execute("""
        INSERT INTO documents (doc_id, title, source, doc_type)
        VALUES (%s, %s, %s, %s)
    """, (doc_id, parsed_doc["title"], parsed_doc["source"], parsed_doc["doc_type"]))

    # Insert sections
    for section in parsed_doc["sections"]:
        section_id = str(uuid.uuid4())
        section_id_map[section["title"]] = section_id
        cur.execute("""
            INSERT INTO sections (section_id, doc_id, title, level)
            VALUES (%s, %s, %s, %s)
        """, (section_id, doc_id, section["title"], section["level"]))

    # Insert chunks
    chunk_id_list = []
    for chunk, embedding in zip(all_chunks, chunk_embeddings):
        chunk_id = str(uuid.uuid4())
        chunk_id_list.append(chunk_id)
        section_id = section_id_map[chunk["section_title"]]
        cur.execute("""
            INSERT INTO chunks
                (chunk_id, doc_id, section_id, content, embedding,
                 embedding_model_version, modality, token_count,
                 metadata)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            chunk_id, doc_id, section_id,
            chunk["content"], embedding,
            settings.embedding_model_version,
            MODALITY_TEXT,
            chunk["token_count"],
            json.dumps({"section_title": chunk["section_title"]}),
        ))

    # Insert assets
    asset_id_list = []
    for img in parsed_doc["images"]:
        asset_id = str(uuid.uuid4())
        asset_id_list.append(asset_id)
        asset_emb = model.encode(img["caption"], normalize_embeddings=True)
        cur.execute("""
            INSERT INTO assets (asset_id, doc_id, type, content, embedding)
            VALUES (%s, %s, %s, %s, %s)
        """, (asset_id, doc_id, MODALITY_IMAGE, img["caption"], asset_emb))

    # Link first chunk to first asset
    cur.execute("""
        INSERT INTO relationships (source_id, target_id, relation_type)
        VALUES (%s, %s, %s)
    """, (chunk_id_list[0], asset_id_list[0], RELATION_CHUNK_TO_ASSET))

    conn.commit()

print(f"Stored: 1 document, {len(section_id_map)} sections, {len(chunk_id_list)} chunks, {len(asset_id_list)} assets, 1 relationship")

## 7. Verify Stored Data

In [ ]:
import pandas as pd

with conn.cursor() as cur:
    cur.execute("""
        SELECT c.chunk_id, s.title AS section, c.token_count,
               c.embedding_model_version, c.tsv IS NOT NULL AS tsv_ok,
               c.content
        FROM chunks c
        JOIN sections s ON c.section_id = s.section_id
        WHERE c.doc_id = %s
        ORDER BY s.level, c.created_at;
    """, (doc_id,))
    rows = cur.fetchall()

df = pd.DataFrame(rows, columns=["chunk_id", "section", "tokens", "model_version", "tsv_ok", "content"])
df["content_preview"] = df["content"].str[:60] + "..."
print(df[["section", "tokens", "model_version", "tsv_ok", "content_preview"]].to_string(index=False))

In [ ]:
# Verify BM25 search works on stored chunks
with conn.cursor() as cur:
    cur.execute("""
        SELECT content, ts_rank(tsv, plainto_tsquery('english', 'vector search')) AS score
        FROM chunks
        WHERE doc_id = %s
        ORDER BY score DESC
        LIMIT 3;
    """, (doc_id,))
    bm25_results = cur.fetchall()

print("\nBM25 search — 'vector search':")
for content, score in bm25_results:
    print(f"  score={score:.4f}  {content[:80]}")

In [ ]:
# Verify pgvector search works
query = "How does approximate nearest neighbour search work?"
q_emb = model.encode(
    "Represent this sentence for searching relevant passages: " + query,
    normalize_embeddings=True
)

with conn.cursor() as cur:
    cur.execute("""
        SELECT content, (embedding <=> %s) AS distance
        FROM chunks
        WHERE doc_id = %s
        ORDER BY distance
        LIMIT 3;
    """, (q_emb, doc_id))
    vec_results = cur.fetchall()

print(f"\nVector search — '{query}':")
for content, dist in vec_results:
    print(f"  dist={dist:.4f}  {content[:80]}")

In [ ]:
# Verify relationship
with conn.cursor() as cur:
    cur.execute("""
        SELECT r.source_id, r.target_id, r.relation_type,
               a.content AS asset_content
        FROM relationships r
        JOIN assets a ON r.target_id = a.asset_id
        WHERE r.source_id = %s;
    """, (chunk_id_list[0],))
    rels = cur.fetchall()

print("\nRelationships:")
for source, target, rel_type, asset_content in rels:
    print(f"  {rel_type}: chunk → asset")
    print(f"  Asset: {asset_content}")

## 8. Pipeline Timing Summary

In [ ]:
print("Ingestion pipeline completed successfully")
print()
print(f"  Document : {parsed_doc['title']}")
print(f"  Sections : {len(section_id_map)}")
print(f"  Chunks   : {len(chunk_id_list)}")
print(f"  Assets   : {len(asset_id_list)}")
print(f"  Relations: 1")
print()
print("All checks passed:")
print("  ✅ Parsed")
print("  ✅ Chunked with semantic boundaries")
print("  ✅ Embedded (1024-dim, L2-normalised)")
print("  ✅ Stored in Postgres")
print("  ✅ tsv trigger fired (BM25 ready)")
print("  ✅ Vector search working")
print("  ✅ Relationships stored")

## 9. Cleanup

In [ ]:
with conn.cursor() as cur:
    cur.execute("DELETE FROM documents WHERE doc_id = %s", (doc_id,))   # cascades
    conn.commit()
conn.close()
print("Test data cleaned up. Phase 2 notebooks complete.")
print("\n→ Ready for Phase 2 modular code implementation.")